# 11 — Model Training v2
Trains Poisson GLM on `features_v2.csv`. Saves `home_model_v2.pkl` and `away_model_v2.pkl`.

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.formula.api import glm
from statsmodels.genmod.families import Poisson
from scipy.stats import poisson
import pickle

In [ ]:
df = pd.read_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\features_v2.csv")
print(df.shape)
df.head()

In [ ]:
model_df = df.dropna(subset=[
    'home_score', 'away_score',
    'home_elo', 'away_elo',
    'elo_diff', 'fifa_rank_diff',
    'tournament_weight', 'neutral'
]).copy()

model_df['neutral'] = model_df['neutral'].astype(int)
print(f"Training rows: {len(model_df)}")

In [ ]:
home_model = glm(
    'home_score ~ home_elo + away_elo + elo_diff + neutral + tournament_weight + fifa_rank_diff',
    data=model_df,
    family=Poisson()
).fit()

print(home_model.summary())

In [ ]:
away_model = glm(
    'away_score ~ home_elo + away_elo + elo_diff + neutral + tournament_weight + fifa_rank_diff',
    data=model_df,
    family=Poisson()
).fit()

print(away_model.summary())

In [ ]:
print("Home model pseudo R²:", round(home_model.pseudo_rsquared(), 3))
print("Away model pseudo R²:", round(away_model.pseudo_rsquared(), 3))

print("\nHome model coefficients:")
print(home_model.params.round(4))

print("\nAway model coefficients:")
print(away_model.params.round(4))

In [ ]:
with open(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\home_model_v2.pkl", 'wb') as f:
    pickle.dump(home_model, f)

with open(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\away_model_v2.pkl", 'wb') as f:
    pickle.dump(away_model, f)

print("Both v2 models saved!")

In [ ]:
# Spain vs Argentina on neutral ground
test_match = pd.DataFrame([{
    'home_elo': 1916.51,   # Spain v2 Elo
    'away_elo': 1900.96,   # Argentina v2 Elo
    'elo_diff': 1916.51 - 1900.96,
    'neutral': 1,
    'tournament_weight': 5.0,
    'fifa_rank_diff': 2 - 3
}])

home_goals = home_model.predict(test_match)[0]
away_goals = away_model.predict(test_match)[0]

print(f"Spain expected goals:     {home_goals:.2f}")
print(f"Argentina expected goals: {away_goals:.2f}")

def match_probabilities(home_xg, away_xg, max_goals=10):
    home_win, draw, away_win = 0, 0, 0
    for i in range(max_goals):
        for j in range(max_goals):
            p = poisson.pmf(i, home_xg) * poisson.pmf(j, away_xg)
            if i > j: home_win += p
            elif i == j: draw += p
            else: away_win += p
    return round(home_win, 4), round(draw, 4), round(away_win, 4)

hw, d, aw = match_probabilities(home_goals, away_goals)
print(f"\nSpain win:     {hw*100:.1f}%")
print(f"Draw:          {d*100:.1f}%")
print(f"Argentina win: {aw*100:.1f}%")

In [ ]:
feature_cols = ['home_elo', 'away_elo', 'elo_diff', 'neutral', 'tournament_weight', 'fifa_rank_diff']

with open(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\feature_cols_v2.pkl", 'wb') as f:
    pickle.dump(feature_cols, f)

print("Feature columns saved!")